# Actividad: reconocimiento de Captchas con PyTorch

Un **captcha** es una imagen con unos caracteres distorsionados que una persona puede leer fácilmente, pero que a un programa le cuesta. En esta actividad construimos una red convolucional que **lee los 5 caracteres de un captcha**.

La base de datos es [Captcha](https://www.researchgate.net/publication/248380891_CAPTCHA_dataset): 1070 imágenes en escala de grises (una tiene un nombre de archivo atípico y se descarta, quedan 1069) de 50×200 píxeles. El nombre de cada archivo es el texto del captcha (por ejemplo, `226md.png` contiene `226md`).

**Qué es distinto frente a lo que hemos visto.** Hasta ahora una imagen tenía *una* etiqueta (gato o perro). Aquí una imagen tiene *cinco* etiquetas, una por posición, y cada una puede ser una de 36 opciones (26 letras minúsculas + 10 dígitos). Por eso la red va a producir **5 predicciones de 36 clases cada una**: una salida de forma `(batch, 5, 36)`.

**Cómo está organizada la actividad.** Las etapas 1, 3 y 4 ya están listas. Tu trabajo está en la **Etapa 2**: definir la red que planteaste en la Reflexión. Busca las celdas marcadas con `TODO`.

## Etapa 1: Carga de los datos

In [ ]:
import os
import string
import zipfile

import gdown
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Dispositivo:", device)

Se descarga la base de datos (un `.zip` de ~10 MB) y se descomprime en la carpeta `Data`.

In [ ]:
if not os.path.exists("Data/captcha_dataset"):
    gdown.download(id="1EcTWqMW8tLYQ32_MXg2m_xrWo1w0AqXf", output="Data.zip")
    with zipfile.ZipFile("Data.zip") as z:
        z.extractall("Data")

IMG_DIR = "Data/captcha_dataset/samples"
print(len(os.listdir(IMG_DIR)), "imágenes")

Cada carácter se convierte en un número entre 0 y 35 (`a` → 0, `b` → 1, …, `z` → 25, `0` → 26, …, `9` → 35). Las imágenes se reescalan a valores entre 0 y 1 y quedan con forma `(1, H, W)` = `(1, 50, 200)`, que es el formato `(canales, alto, ancho)` que usa PyTorch.

In [ ]:
H, W = 50, 200
N_POS = 5                                              # posiciones (caracteres) por captcha
characters = string.ascii_lowercase + string.digits    # 36 símbolos posibles
N_CHAR = len(characters)


def load_data():
    images, labels, texts = [], [], []
    for fname in sorted(os.listdir(IMG_DIR)):
        text = os.path.splitext(fname)[0]
        if len(text) != N_POS:                         # se ignoran archivos con etiqueta rara
            continue
        img = Image.open(os.path.join(IMG_DIR, fname)).convert("L").resize((W, H))
        images.append(np.asarray(img, dtype=np.float32) / 255.0)
        labels.append([characters.index(c) for c in text])
        texts.append(text)
    X = torch.tensor(np.stack(images)).unsqueeze(1)    # (N, 1, H, W)
    y = torch.tensor(labels)                           # (N, 5) con enteros 0..35
    return X, y, texts


X, y, texts = load_data()
print(X.shape, y.shape)

fig, axes = plt.subplots(3, 2, figsize=(8, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X[i, 0], cmap="gray")
    ax.set_title(f"Etiqueta: {texts[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

Se separan los datos: las primeras 970 imágenes son de entrenamiento y las restantes (99) de prueba.

In [ ]:
n_train = 970
x_train, y_train = X[:n_train], y[:n_train]
x_test, y_test = X[n_train:], y[n_train:]

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=100)
print(len(x_train), "para entrenar,", len(x_test), "para probar")

## Etapa 2: Define la red (TODO)

Aquí va la solución que propusiste en la Reflexión. Reglas del problema:

- La entrada es un batch de imágenes de forma `(batch, 1, 50, 200)`.
- La salida **debe** tener forma `(batch, 5, 36)`: para cada una de las 5 posiciones, 36 *logits* (uno por símbolo posible). No apliques `softmax`: la función de pérdida lo hace por dentro.
- Solo puedes usar lo visto en el curso: capas convolucionales, pooling, capas lineales, activaciones y las ideas de la unidad (por ejemplo, `Dropout` o `BatchNorm`).

**Pistas** (con ellas la red alcanza entre 80 y 87 % por carácter y entre 25 y 40 % por captcha completo, según la corrida):

- Usa varios bloques `Conv2d` → `BatchNorm2d` → `ReLU`, con `MaxPool2d(2)` solo en algunos (cada pooling reduce a la mitad el alto y el ancho, y el texto es pequeño: no conviene reducir demasiado rápido).
- Termina con **5 cabezas lineales, una por posición** (cada una con `Flatten`, `Dropout` y `Linear` hasta 36 salidas) y apílalas con `torch.stack(salidas, dim=1)`. Una sola cabeza gigante con `5 * 36` salidas y `.view(-1, 5, 36)` también funciona, pero rinde bastante peor.
- Para saber cuántas entradas necesita la primera capa `Linear`, pasa un tensor de prueba `torch.zeros(1, 1, 50, 200)` por las capas convolucionales y mira su `shape`.

In [ ]:
class CaptchaNet(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define las capas (convolucionales, pooling, lineales...)
        raise NotImplementedError("Define las capas de la red en __init__")

    def forward(self, x):
        # x: (batch, 1, 50, 200)  ->  salida: (batch, 5, 36)
        # TODO: aplica las capas y devuelve los logits
        raise NotImplementedError("Escribe el forward de la red")

## Etapa 3: Entrenamiento

La clase `Trainer` ya está completa. Lo único nuevo frente a los notebooks anteriores es la pérdida: `CrossEntropyLoss` espera los logits con forma `(batch, clases)`, así que se aplanan las 5 posiciones, `(batch, 5, 36)` → `(batch * 5, 36)`, y se calcula una sola pérdida sobre todos los caracteres. Se reportan dos métricas:

- **Exactitud por carácter:** qué fracción de caracteres individuales acierta la red.
- **Exactitud por captcha:** qué fracción de captchas acierta **los 5** caracteres (la que de verdad importa).

In [ ]:
class Trainer:
    def __init__(self, model, lr=1e-3):
        self.model = model.to(device)
        self.loss_fn = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        self.history = {"loss": [], "test_loss": [], "char_acc": [], "captcha_acc": []}

    def compute_loss(self, logits, y):
        return self.loss_fn(logits.reshape(-1, N_CHAR), y.reshape(-1))

    def train_epoch(self, loader):
        self.model.train()
        total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            self.optimizer.zero_grad()
            loss = self.compute_loss(self.model(xb), yb)
            loss.backward()
            self.optimizer.step()
            total += loss.item() * len(xb)
        return total / len(loader.dataset)

    @torch.no_grad()
    def evaluate(self, loader):
        self.model.eval()
        total, char_ok, captcha_ok = 0.0, 0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = self.model(xb)
            total += self.compute_loss(logits, yb).item() * len(xb)
            hits = logits.argmax(dim=2) == yb              # (batch, 5) True si acierta
            char_ok += hits.sum().item()
            captcha_ok += hits.all(dim=1).sum().item()
        n = len(loader.dataset)
        return total / n, char_ok / (n * N_POS), captcha_ok / n

    def fit(self, train_loader, test_loader, epochs=60):
        for epoch in range(1, epochs + 1):
            loss = self.train_epoch(train_loader)
            test_loss, char_acc, captcha_acc = self.evaluate(test_loader)
            for k, v in zip(self.history, (loss, test_loss, char_acc, captcha_acc)):
                self.history[k].append(v)
            print(f"Época {epoch:3d} | pérdida {loss:.3f} | prueba {test_loss:.3f} | "
                  f"carácter {char_acc:.1%} | captcha {captcha_acc:.1%}")

    @torch.no_grad()
    def predict(self, x):
        self.model.eval()
        pred = self.model(x.to(device)).argmax(dim=2).cpu()
        return ["".join(characters[i] for i in row) for row in pred]

In [ ]:
# TODO: ajusta epochs y lr si lo necesitas
trainer = Trainer(CaptchaNet(), lr=1e-3)
trainer.fit(train_loader, test_loader, epochs=60)

## Etapa 4: Resultados

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(trainer.history["loss"], label="entrenamiento")
ax[0].plot(trainer.history["test_loss"], label="prueba")
ax[0].set_title("Pérdida"); ax[0].set_xlabel("época"); ax[0].legend()
ax[1].plot(trainer.history["char_acc"], label="por carácter")
ax[1].plot(trainer.history["captcha_acc"], label="por captcha")
ax[1].set_title("Exactitud en prueba"); ax[1].set_xlabel("época"); ax[1].legend()
plt.tight_layout(); plt.show()

preds = trainer.predict(x_test[:6])
fig, axes = plt.subplots(3, 2, figsize=(8, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_test[i, 0], cmap="gray")
    ax.set_title(f"Real: {texts[n_train + i]} | Predicción: {preds[i]}")
    ax.axis("off")
plt.tight_layout(); plt.show()

## Para comentar en el debate «Consolidación del aprendizaje»

1. ¿Qué exactitud por carácter y por captcha obtuviste? (con una buena red, ~80-87 % por carácter y ~25-40 % por captcha) ¿Por qué la exactitud por captcha es siempre menor?
2. ¿Qué diferencia hay entre la pérdida de entrenamiento y la de prueba? ¿Hay sobreajuste? ¿Qué harías para reducirlo?
3. ¿Qué capas elegiste y por qué? ¿Cómo organizaste la salida `(batch, 5, 36)`?
4. ¿Qué errores comete la red? ¿Confunde algunos caracteres (por ejemplo, `m` y `n`)?